# NB_lab_test_trends

Builds `adb_qnxthealth.gold.lab_test_trends` — one row per test type per calendar month with test volumes, abnormal/critical rates, and month-over-month trend via LAG(). Full rebuild every run.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_qnxthealth.gold.lab_test_trends AS

WITH monthly_agg AS (
  SELECT
    lr.test_name,
    YEAR(lr.test_date)                                                    AS year,
    MONTH(lr.test_date)                                                   AS month,
    COUNT(*)                                                              AS total_tests,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)            AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)            AS critical_test_count
  FROM adb_qnxthealth.silver.lab_results lr
  GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date)
),

with_rates AS (
  SELECT
    ma.*,
    CAST(ROUND(ma.abnormal_test_count * 100.0 / ma.total_tests, 2) AS DECIMAL(5,2))  AS abnormal_rate_pct,
    CAST(ROUND(ma.critical_test_count * 100.0 / ma.total_tests, 2) AS DECIMAL(5,2))  AS critical_rate_pct
  FROM monthly_agg ma
),

with_trend AS (
  SELECT
    wr.*,
    CAST(
      LAG(wr.abnormal_rate_pct) OVER (PARTITION BY wr.test_name ORDER BY wr.year, wr.month)
      AS DECIMAL(5,2)
    )                                                                     AS prev_month_abnormal_rate_pct
  FROM with_rates wr
)

SELECT
  wt.test_name,
  wt.year,
  wt.month,
  c.month_name,
  c.quarter_name,
  wt.total_tests,
  wt.abnormal_test_count,
  wt.critical_test_count,
  wt.abnormal_rate_pct,
  wt.critical_rate_pct,
  wt.prev_month_abnormal_rate_pct,
  CAST(
    CASE
      WHEN wt.prev_month_abnormal_rate_pct IS NULL THEN NULL
      ELSE ROUND(wt.abnormal_rate_pct - wt.prev_month_abnormal_rate_pct, 2)
    END AS DECIMAL(6,2)
  )                                                                       AS abnormal_rate_change_pct

FROM      with_trend wt
LEFT JOIN adb_qnxthealth.silver.calendar c
       ON c.date_key = CAST(date_format(DATE_TRUNC('month', MAKE_DATE(wt.year, wt.month, 1)), 'yyyyMMdd') AS INT)
ORDER BY  wt.test_name, wt.year, wt.month;

In [0]:
%skip
%sql
SELECT count(*) AS total_rows,
       count(DISTINCT test_name) AS unique_tests,
       min(CONCAT(year, '-', LPAD(month, 2, '0'))) AS earliest_month,
       max(CONCAT(year, '-', LPAD(month, 2, '0'))) AS latest_month,
       avg(abnormal_rate_pct) AS avg_abnormal_rate,
       count_if(prev_month_abnormal_rate_pct IS NOT NULL) AS rows_with_trend
FROM   adb_qnxthealth.gold.lab_test_trends;

In [0]:
record_count = spark.table("adb_qnxthealth.gold.lab_test_trends").count()
dbutils.notebook.exit(str(record_count))